In [1]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from scipy import stats as st

# Preparacao dos Dados

In [2]:
df_telecom_clients = pd.read_csv('../data/telecom_clients.csv')
df_telecom_dataset_new = pd.read_csv('../data/telecom_dataset_new.csv')

In [3]:
df_telecom_clients.head()

,user_id,tariff_plan,date_start
0,166713,A,2019-08-15
1,166901,A,2019-08-23
2,168527,A,2019-10-29
3,167097,A,2019-09-01
4,168193,A,2019-10-16


In [4]:
df_telecom_dataset_new.head()

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.0,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.0,True,3,0,25


In [5]:
df_telecom_clients.info()

<class 'pandas.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      732 non-null    int64
 1   tariff_plan  732 non-null    str  
 2   date_start   732 non-null    str  
dtypes: int64(1), str(2)
memory usage: 17.3 KB


In [6]:
df_telecom_dataset_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  str    
 2   direction            53902 non-null  str    
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(1), str(2)
memory usage: 3.3+ MB


In [7]:
df_telecom_clients['date_start'] = pd.to_datetime(df_telecom_clients['date_start'])
df_telecom_dataset_new['date'] = pd.to_datetime(df_telecom_dataset_new['date'])

In [8]:
df_telecom_clients.info()
df_telecom_dataset_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      732 non-null    int64         
 1   tariff_plan  732 non-null    str           
 2   date_start   732 non-null    datetime64[us]
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 17.3 KB
<class 'pandas.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype                    
---  ------               --------------  -----                    
 0   user_id              53902 non-null  int64                    
 1   date                 53902 non-null  datetime64[us, UTC+03:00]
 2   direction            53902 non-null  str                      
 3   internal             53785 non-null  object                   
 4   operator_id          45730 non-null  float64                  
 5   is_missed_call    

In [9]:
df_telecom_dataset_new['internal'].value_counts(dropna=False)

internal
False    47621
True      6164
NaN        117
Name: count, dtype: int64

In [10]:
df_telecom_dataset_new['internal'] = (
    df_telecom_dataset_new['internal'].astype('boolean')
)
df_telecom_dataset_new['internal'].dtype

BooleanDtype

In [11]:
df_telecom_dataset_new['internal'].value_counts(dropna=False)

internal
False    47621
True      6164
<NA>       117
Name: count, dtype: Int64

In [12]:
df_telecom_dataset_new['operator_id'].dropna().mod(1).value_counts()

operator_id
0.0    45730
Name: count, dtype: int64

In [13]:
df_telecom_dataset_new['operator_id'] = (
    df_telecom_dataset_new['operator_id'].astype('Int64')
)
df_telecom_dataset_new['operator_id'].dtype
df_telecom_dataset_new['operator_id'].isna().sum()

np.int64(8172)

### Conversao e adequacao dos tipos de dados
- A coluna `date` estava inicialmente como texto (str). Ela foi convertida para o tipo datetime, permitindo realizar analises temporais. 
- A coluna `internal` estava inicialmente como object e continha valores True, False e valores ausentes. Apos verificar os valores existentes, ela foi convertida para o tipo anulavel boolean do pandas, preservando os valores ausentes sem atribuir classificacoes arbitrarias.
- A coluna `operator_id` estava como float devido a presenca de valores ausentes. A verificacao dos 45.730 valores nulos confirmou que todos eram numeros inteiros, sem casas decimais. Por isso, a coluna foi convertida para o tipo anulavel Int64, que permite representar identificadores inteiros e manter os valores ausentes. 

### Verificacao de consistencia

In [14]:
df_telecom_dataset_new[
    ['calls_count', 'call_duration', 'total_call_duration']
].describe()

,calls_count,call_duration,total_call_duration
count,53902.000000,53902.000000,53902.000000
mean,16.451245,866.684427,1157.133297
std,62.917170,3731.791202,4403.468763
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,47.000000
50%,4.000000,38.000000,210.000000
75%,12.000000,572.000000,902.000000
max,4817.000000,144395.000000,166155.000000


In [15]:
consistency_check = pd.Series({
    'calls_count <= 0': (df_telecom_dataset_new['calls_count'] <= 0).sum(),
    'call_duration < 0': (df_telecom_dataset_new['call_duration'] < 0).sum(),
    'total_call_duration < 0': (df_telecom_dataset_new['total_call_duration'] < 0).sum(),
    'total_call_duration < call_duration': (
        df_telecom_dataset_new['total_call_duration'] < df_telecom_dataset_new['call_duration']
    ).sum()
})

consistency_check

calls_count <= 0                       0
call_duration < 0                      0
total_call_duration < 0                0
total_call_duration < call_duration    0
dtype: int64

### Verificacao de consistencia das contagens e duracoes
- Foram realizadas quatro verificacoes de consistencia na base de chamadas, contagens menores ou iguais a zero, duracoes de conversa negativas, duracoes totais negativas e registros em que a duracao total fosse inferior a duracao da conversa. 
- Nenhum registro apresentou essas inconsistencias. A coluna `calls_count` contem valores positivos, as duracoes nao apresentam valores negativos e a duracao total e sempre maior ou igual a duracao da conversa. 

- Agora vamos investigar os valores extremos que apareceram em `calls_count`(4.817), `call_duration`(144.395) e `total_call_duration`(166.155).

In [16]:
df_telecom_dataset_new.nlargest(10, 'calls_count')[
    [
        'user_id',
        'date',
        'direction',
        'operator_id',
        'is_missed_call',
        'calls_count',
        'call_duration',
        'total_call_duration'
    ]
]

,user_id,date,direction,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
40733,167827,2019-11-11 00:00:00+03:00,out,929428,True,4817,0,5529
40671,167827,2019-10-31 00:00:00+03:00,out,929428,True,2614,0,45312
37070,167626,2019-10-07 00:00:00+03:00,in,<NA>,True,2168,0,2361
37102,167626,2019-10-08 00:00:00+03:00,in,<NA>,True,1917,0,2044
37553,167626,2019-10-15 00:00:00+03:00,in,<NA>,True,1914,0,2063
37308,167626,2019-10-11 00:00:00+03:00,in,<NA>,True,1705,0,2455
37343,167626,2019-10-11 00:00:00+03:00,in,<NA>,True,1705,0,2455
50837,168252,2019-11-25 00:00:00+03:00,in,<NA>,True,1679,0,42251
36913,167626,2019-10-04 00:00:00+03:00,in,<NA>,True,1641,0,1210
50888,168252,2019-11-26 00:00:00+03:00,in,<NA>,True,1590,0,42667


In [17]:
df_telecom_dataset_new[
    df_telecom_dataset_new['calls_count'] >= 1500
].duplicated(keep=False).sum()

np.int64(2)

In [18]:
high_calls = df_telecom_dataset_new[
    df_telecom_dataset_new['calls_count'] >= 1500
]
high_calls[
    high_calls.duplicated(keep=False)
].sort_values('calls_count', ascending=False)

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
37308,167626,2019-10-11 00:00:00+03:00,in,False,<NA>,True,1705,0,2455
37343,167626,2019-10-11 00:00:00+03:00,in,False,<NA>,True,1705,0,2455


In [19]:
df_telecom_dataset_new[
    df_telecom_dataset_new['calls_count'] >= 1500
].groupby([
    'direction', 'is_missed_call'],
    dropna=False
).agg(
    register_total = ('user_id', 'size'),
    calls_total = ('calls_count', 'sum'),
    max_calls_count = ('calls_count', 'max'),
)

,,register_total,calls_total,max_calls_count
direction,is_missed_call,,,
in,True,8,14319,2168
out,True,4,10493,4817


### Investigacao de valores extremos em calls_count
- A analise dos dez maiores valores de `calls_count` identificou registros com contagens elevadas de chamadas. Para investigar esses valores, selecionamos os registros com pelo menos 1500 chamadas. 
- Foram encontrados 12 registros nessa condicao, todos classificados como chamadas perdidas. Oito registros correspondem a chamadas recebidas, totalizando 14.319 chamadas, e quatro correspondem a chamadas efetuadas, totalizado 10.493 chamadas. A maior contagem individual foi de 4.817 chamadas. 
- Como a base contem registros que podem representar multiplas chamadas agregadas, valores elevados nao foram considerados erros automaticamente. Eles sera preservados, e os resultados serao interpretados considerando a estrutura agregada dos dados. 

### Investigate the null values

In [20]:
df_telecom_dataset_new.groupby(
    ['direction', 'is_missed_call'],
    dropna=False
).agg(
    register_total = ('user_id', 'size'),
    operator_missing = ('operator_id', lambda s: s.isna().sum())
)

register_total  operator_missing
direction is_missed_call                                  
in        False                    13342                73
          True                      8643              7899
out       False                    16992                49
          True                     14925               151

In [21]:
df_telecom_dataset_new.groupby('direction').agg(
    register_total = ('user_id', 'size'),
    operator_missing = ('operator_id', 'count')
).assign(
    operator_absent = lambda df:
        df['register_total'] - df['operator_missing']
)

,register_total,operator_missing,operator_absent
direction,,,
in,21985,14013,7972
out,31917,31717,200


In [22]:
missing_values = pd.DataFrame(
    {
    'missing_count': df_telecom_dataset_new.isna().sum(),
    'missing_percentage': (
        df_telecom_dataset_new.isna().mean() * 100
    ).round(2)
})
missing_values.sort_values(by='missing_count', ascending=False)

,missing_count,missing_percentage
operator_id,8172,15.16
internal,117,0.22
user_id,0,0.00
date,0,0.00
direction,0,0.00
is_missed_call,0,0.00
calls_count,0,0.00
call_duration,0,0.00
total_call_duration,0,0.00


In [23]:
df_telecom_dataset_new[
    df_telecom_dataset_new['internal'].isna()
].groupby(
    ['direction', 'is_missed_call'],
    dropna=False
).agg(
    register_total = ('user_id', 'size'),
    calls_total = ('calls_count', 'sum')
)

register_total  calls_total
direction is_missed_call                             
in        False                       59           64
          True                        56          164
out       True                         2            6

### Analise de valores ausentes
- A analise identificou 8.172 valores ausentes na coluna `operator_id`, correspondentes a 15,16% dos registros, e 117 valores ausentes na coluna internal, equivalentes a 0,22% da base original.
- A ausencia de `operator_id` esta concentrada nas chamadas recebidas, principalmente naquelas classificadas como perdidas. Como nao ha informacoes suficientes esses valores nao serao preenchidos. Os registros serao preservados na analise geral e nao serao utilizados nos indicadores que dependem da identificacao individual do operador. 
- O valores ausentes em internal representam 234 chamadas, de acordo com a soma da coluna `calls_count`. 
- A analise por categoria mostrou que os valores ausentes estao presentes em chamadas recebidas tanto perdidas quanto nao perdidas, e em dois registros de chamadas efetuadas perdidas. Nao foram encontrados valores ausentes nessa coluna entre as chamadas efetuadas nao perdidas. 

### Investigate the duplicate

In [24]:
duplicate_count = df_telecom_clients.duplicated().sum()

duplicate_percentage = duplicate_count / len(df_telecom_clients) * 100

print("Duplicatas:", duplicate_count)
print("Percentual:", duplicate_percentage)

Duplicatas: 0
Percentual: 0.0


In [25]:
df_telecom_clients[df_telecom_clients.duplicated(keep=False)].head(10)

,user_id,tariff_plan,date_start


In [26]:
duplicate_clients = (
    df_telecom_clients['user_id']
    .duplicated()
    .sum()
)
print(f"Number of duplicate user_id in df_telecom_clients: {duplicate_clients}")
print(f"Number of duplicate rows considering user_id: "
      f"{df_telecom_clients.duplicated(subset=['user_id'], keep=False).sum()}")
print(f"Number of unique users in df_telecom_clients: {df_telecom_clients['user_id'].nunique()}")
print(f"Total number of lines in df_telecom_clients: {len(df_telecom_clients)}")

Number of duplicate user_id in df_telecom_clients: 0
Number of duplicate rows considering user_id: 0
Number of unique users in df_telecom_clients: 732
Total number of lines in df_telecom_clients: 732


In [27]:
duplicate_count_dataset = df_telecom_dataset_new.duplicated().sum()

duplicate_percentage_dataset = duplicate_count_dataset / len(df_telecom_dataset_new) * 100

print("Duplicatas:", duplicate_count_dataset)
print("Percentual:", duplicate_percentage_dataset)

Duplicatas: 4900
Percentual: 9.090571778412675


- 4,900 duplicate records were found, representing approximately 9.09% of the database.

In [28]:
df_telecom_dataset_new[df_telecom_dataset_new.duplicated(keep=False)].head(10)

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
6,166377,2019-08-05 00:00:00+03:00,out,False,880020,True,8,0,50
8,166377,2019-08-05 00:00:00+03:00,out,False,880020,True,8,0,50
27,166377,2019-08-12 00:00:00+03:00,in,False,<NA>,True,2,0,34
28,166377,2019-08-12 00:00:00+03:00,in,False,<NA>,True,2,0,34
38,166377,2019-08-14 00:00:00+03:00,in,False,<NA>,True,1,0,3
43,166377,2019-08-14 00:00:00+03:00,out,False,880026,False,10,1567,1654
44,166377,2019-08-14 00:00:00+03:00,out,False,880026,False,10,1567,1654
45,166377,2019-08-14 00:00:00+03:00,in,False,<NA>,True,1,0,3
46,166377,2019-08-15 00:00:00+03:00,out,False,880026,False,11,1413,1473
51,166377,2019-08-15 00:00:00+03:00,out,False,880026,False,11,1413,1473


In [29]:
df_without_duplicates = df_telecom_dataset_new.drop_duplicates()

print('Original records: ', len(df_telecom_dataset_new))

print('Records without duplicates: ', len(df_without_duplicates))

missed_calls_before = df_telecom_dataset_new.loc[
    df_telecom_dataset_new['is_missed_call'] == True,
        'calls_count'
    ].sum()



missed_calls_after = df_without_duplicates.loc[
    df_without_duplicates['is_missed_call'] == True,
        'calls_count'
    ].sum()

print("Missed calls before:", missed_calls_before)
print("Missed calls after:", missed_calls_after)
print("Difference:", missed_calls_before - missed_calls_after)

Original records:  53902
Records without duplicates:  49002
Missed calls before: 412305
Missed calls after: 375918
Difference: 36387


- O total de chamadas perdidas diminuiu 36.387, aproximadamente 8,83% do total original de chamadas perdidas, sendo diferencas entre os dois cenarios calculados. Sendo significativo para a nossa analise: a presenca dessas duplicatas altera materialmente uma metrica importante do projeto. 

In [30]:
duplicate_rows = df_telecom_dataset_new[
    df_telecom_dataset_new.duplicated(keep=False)
]

duplicate_rows.groupby('direction').agg(
    duplicate_records=('user_id', 'size'),
    missed_calls=('calls_count', 'sum'),
    operators=('operator_id', 'nunique')
)

missed_by_operator_before = (
    df_telecom_dataset_new[
        (df_telecom_dataset_new['direction'] == 'in') &
        (df_telecom_dataset_new['is_missed_call'] == True)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
)

missed_by_operator_after = (
    df_without_duplicates[
        (df_without_duplicates['direction'] == 'in') &
        (df_without_duplicates['is_missed_call'] == True)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
)

comparison = pd.concat([
    missed_by_operator_before.rename('before'),
    missed_by_operator_after.rename('after')
], axis=1).fillna(0)

comparison['difference'] = comparison['before'] - comparison['after']

comparison.sort_values(by='difference', ascending=False).head(10)

,before,after,difference
operator_id,,,
944216,32,24,8
944226,38,30,8
885876,24,20,4
937860,8,5,3
913942,55,52,3
944220,21,18,3
951508,24,21,3
940658,13,10,3
919206,12,9,3


In [31]:
comparison['percentage_difference'] = (
    comparison['difference']
    .div(comparison['before'].replace(0, float('nan')))
    .mul(100)
)

print('Largest absolute difference:', comparison['difference'].max())

print('\nAffected operators:')
print((comparison['difference'] > 0).sum())
print('\nLargest percentage difference:', comparison['percentage_difference'].max())

comparison['rank_before'] = (
    comparison['before']
    .rank(method='min', ascending=False)
)
comparison['rank_after'] = (
    comparison['after']
    .rank(method='min', ascending=False)
)
comparison['rank_change'] = (
comparison['rank_before'] - comparison['rank_after']
)

print('\nLargest rank change:', comparison['rank_change'].abs().max())

Largest absolute difference: 8

Affected operators:
46

Largest percentage difference: 50.0

Largest rank change: 36.0


In [32]:
comparison.sort_values(by='rank_change', key=abs, ascending=False)[
    [
    'before',
    'after',
    'difference',
    'rank_before',
    'rank_after',
    'rank_change'
    ]
].head(10)

,before,after,difference,rank_before,rank_after,rank_change
operator_id,,,,,,
920726,2,1,1,94.0,130.0,-36.0
953464,2,1,1,94.0,130.0,-36.0
919792,2,1,1,94.0,130.0,-36.0
919896,2,1,1,94.0,130.0,-36.0
906294,2,1,1,94.0,130.0,-36.0
905566,2,1,1,94.0,130.0,-36.0
937812,2,1,1,94.0,130.0,-36.0
934430,2,1,1,94.0,130.0,-36.0
911142,3,2,1,68.0,90.0,-22.0


In [33]:
print("Records original: ", len(df_telecom_dataset_new))

print("Exact duplicates: ", df_telecom_dataset_new.duplicated().sum())

print("Records after removing duplicates: ", len(df_telecom_dataset_new.drop_duplicates()))

Records original:  53902
Exact duplicates:  4900
Records after removing duplicates:  49002


### Tratamento de registros duplicados

- Durante a analise exploratoria, foram identificados 4.900 registros duplicados exatos, correspondentes a aproximadamente 9,09% dos registros originais. A remocao dessas duplicatas reduziu o volume de registros de 53.902 para 49.002 e o total de chamadas perdidas de 412.305 para 375.918, uma diferenca de 36.387 chamadas.
- A comparacao por operador mostrou que as maiores mudancas de posicao no ranking ocorreram entre os operadores com poucas chamadas perdidas. Alguns operadores passaram de duas para uma chamada perdida e mudaram 36 posicoes, demonstrando que o ranking e sensivel a pequenas diferencas absolutas.
- A identificacao de duplicatas foi realizada considerando todas as colunas disponiveis. Entretanto, os dados nao possuem um identificador unico de chamada que permita confirmar individualmente a origem de cada registro duplicado. Portanto, nao e possivel afirmar que todas as duplicatas identificadas resultam de uma duplicacao indevida.
- Portanto para a analise principal, serao utilizados os registros sem duplicatas exatas, mantendo-se a base original preservada. Essa decisao se basea na possibilidade de duplicacao indevida, mas reconhece que os dados disponiveis nao permitem confirmar individualmente a origem de cada registro duplicado. 

In [34]:
df_original = df_telecom_dataset_new

df_deduplicated = df_telecom_dataset_new.drop_duplicates()

comparison_missed_calls = pd.DataFrame({
    'original': [
        df_original.loc[
            df_original['is_missed_call'], 'calls_count'
        ].sum()
    ],
    'deduplicated': [
        df_deduplicated.loc[
            df_deduplicated['is_missed_call'], 'calls_count'
        ].sum()
    ]
}, index=['missed_calls_total'])

comparison_missed_calls['difference'] = (
    comparison_missed_calls['original'] - comparison_missed_calls['deduplicated']
)

comparison_missed_calls

,original,deduplicated,difference
missed_calls_total,412305,375918,36387


- A comparacao dos indicadores antes e depois da remocao das duplicatas mostrou uma reducao de 412.305 para 375.918 chamadas perdidas, uma diferenca de 36.387 chamadas, equivalente a aproximadamente 8,83% do total original. Esse resultado demonstra que o tratamento de duplicatas tem impacto relevante nos indicadore agregados. Entretanto, como nao ha um identificador unico de chamada para confirmar a origem dos registros, essa diferenca deve ser interpretada como o impacto da remocao, e nao como uma comprovacao do numero de chamadas efetivamente duplicadas.

In [35]:
df_telecom_analysis = df_telecom_dataset_new.drop_duplicates().copy()
print("Registros para análise:", len(df_telecom_analysis))
print("Duplicatas restantes:", df_telecom_analysis.duplicated().sum())

Registros para análise: 49002
Duplicatas restantes: 0


# Analise Exploratoria de Dados - EDA

In [36]:
df_incoming = df_telecom_analysis[df_telecom_analysis['direction'] == 'in'].copy()
df_incoming.shape

(20003, 9)

#### Calcular chamadas recebidas e perdidas por operador

In [37]:
df_incoming_by_operator = df_incoming[
    df_incoming['operator_id'].notna()
].copy()

operator_metrics = df_incoming_by_operator.groupby('operator_id').agg(
    incoming_calls=('calls_count', 'sum'),
    missed_calls=(
        'calls_count',
        lambda s: s[df_incoming_by_operator.loc[s.index, 'is_missed_call']].sum()
    )
)

operator_metrics['missed_call_rate'] = (
    operator_metrics['missed_calls'] / operator_metrics['incoming_calls'] * 100
)

operator_metrics = operator_metrics.sort_values(by='missed_call_rate', ascending=False)

operator_metrics.head(10)

,incoming_calls,missed_calls,missed_call_rate
operator_id,,,
948756,1,1,100.000000
904344,4,3,75.000000
913886,2,1,50.000000
906392,18,7,38.888889
957922,3,1,33.333333
937432,3,1,33.333333
934098,6,2,33.333333
897872,3,1,33.333333
906412,26,7,26.923077


In [38]:
overall_incoming_calls = df_incoming['calls_count'].sum()

overall_missed_calls = df_incoming.loc[
    df_incoming['is_missed_call'], 'calls_count'
].sum() 

overall_missed_rate = (
    overall_missed_calls / overall_incoming_calls * 100
)

print("Total calls received:", overall_incoming_calls)
print("Total missed calls:", overall_missed_calls)
print("Overall missed call rate: {:.2f}%".format(overall_missed_rate))

Total calls received: 197845
Total missed calls: 104323
Overall missed call rate: 52.73%


#### Comparar os operadores com a taxa geral

In [39]:
operator_metrics['difference_vs_overall_pp'] = (
    operator_metrics['missed_call_rate'] - overall_missed_rate
)

operator_metrics.sort_values(by='difference_vs_overall_pp', ascending=False).head(10)

,incoming_calls,missed_calls,missed_call_rate,difference_vs_overall_pp
operator_id,,,,
948756,1,1,100.000000,47.270338
904344,4,3,75.000000,22.270338
913886,2,1,50.000000,-2.729662
906392,18,7,38.888889,-13.840773
957922,3,1,33.333333,-19.396329
937432,3,1,33.333333,-19.396329
934098,6,2,33.333333,-19.396329
897872,3,1,33.333333,-19.396329
906412,26,7,26.923077,-25.806585


#### Investigar o volume de chamadas por operador

In [40]:
operator_metrics['incoming_calls'].describe()

count     754.000000
mean      124.405836
std       356.450649
min         1.000000
25%         4.000000
50%        17.000000
75%        78.750000
max      4766.000000
Name: incoming_calls, dtype: float64

- Apos a remocao das duplicatas exatas, foi criada uma base de analise com 49.002 registros. A partir dela, foram selecionadas 20.003 linhas correspondentes as chamadas recebidas (direction = in).
- Para avaliar o desempenho individual, foram considerados apenas os registros com `operator_id` identificado. Foram calculados o total de chamadas recebidas, o total de chamadas perdidas e a taxa de chamadas perdidas por operador. Esses indicadores utilizam a soma de `calls_count`, pois cada registro pode representar multiplas chamadas. 
- A analise inicial mostrou operadores com taxas de chamadas perdidas elevadas, incluindo caso de 100% e 75%. Entretanto, esses percentuais foram calculados sobre volumes muito pequenos, como uma chamada recebida ou quatro chamadas recebidas. Por isso, taxas elevafas isoladamente nao sao suficientes para concluir que um operador apresenta desempenho insatisfatorio. 
- A distribuicao do volume de chamadas entre os 754 operadores analisados mostrou uma media de 124,41 chamadas e uma mediana de 17 chamadas por operador. O primeiro quartil foi de 4 chamadas, o terceiro quartil foi de 78,75 chamadas e o maior volume individual foi de 4.766 chamadas. A diferenca entre a media e a mediana indica uma distribuicao assimetrica, com alguns operadores concentrando volumes elevados de chamadas.
- Vale destacar que tambem foi calculada uma taxa geral de chamadas recebidas perdidas de 52,73%, considerando todos os registros de chamadas recebidas, inclusive aqueles sem operador identificado. Como os indicadores individuais incluem somente chamadas atribuidas a operadores, essa taxa geral nao deve ser utilizada como referencia direta sem considerar a diferenca entre as populacoes analisadas. 

#### Calcular a taxa de referencia

In [41]:
identified_incoming_calls = (
    operator_metrics['incoming_calls'].sum()
)

identified_missed_calls = (
    operator_metrics['missed_calls'].sum()
)

identified_missed_rate = (
    identified_missed_calls / identified_incoming_calls * 100
)

print("Incoming calls with identified operator:", identified_incoming_calls)

print("Missed calls with identified operator:",identified_missed_calls)

print("Missed call rate with identified operator: {:.2f}%".format(identified_missed_rate))   

Incoming calls with identified operator: 93802
Missed calls with identified operator: 926
Missed call rate with identified operator: 0.99%


In [42]:
df_incoming.groupby(
    [
        df_incoming['operator_id'].notna().rename(
            'operator_identified'
        ),
        'is_missed_call'
    ]
).agg(
    register_total=('user_id', 'size'),
    calls_total=('calls_count', 'sum')
)

register_total  calls_total
operator_identified is_missed_call                             
False               False                       68          646
                    True                      7202       103397
True                False                    12048        92876
                    True                       685          926

In [43]:
incoming_by_id_status = df_incoming.assign(
    operator_identified = df_incoming['operator_id'].notna()
).groupby('operator_identified').apply(
    lambda group: pd.Series({
        'register_total': len(group),
        'calls_total': group['calls_count'].sum(),
        'missed_calls': group.loc[
            group['is_missed_call'], 'calls_count'
        ].sum(),
        'missed_call_rate_pct': (
            group.loc[
                group['is_missed_call'], 'calls_count'
            ].sum()
            / group['calls_count'].sum()
            * 100
        )
    }),
    include_groups=False
)
incoming_by_id_status

,register_total,calls_total,missed_calls,missed_call_rate_pct
operator_identified,,,,
False,7270.0,104043.0,103397.0,99.379103
True,12733.0,93802.0,926.0,0.987186


In [44]:
df_incoming.assign(
    operator_identified=df_incoming['operator_id'].notna()
).groupby(
    ['operator_identified', 'is_missed_call']
).size().unstack(fill_value=0)

is_missed_call,False,True
operator_identified,,
False,68,7202
True,12048,685


In [45]:
df_incoming[df_incoming['operator_id'].isna()].groupby(
    'is_missed_call'
)['calls_count'].agg(
    register_total='count',
    calls_total='sum',
    mean_calls='mean',
    median_calls='median',
    max_calls='max'
)

,register_total,calls_total,mean_calls,median_calls,max_calls
is_missed_call,,,,,
False,68,646,9.500000,2.0,52
True,7202,103397,14.356706,2.0,2168


- Os registros foram separados em dois grupos de acordo com a presença de `operator_id`. Os resultados mostram uma diferença expressiva entre os grupos. Aproximadamente 99,38% do volume de chamadas no grupo sem operador identificado está marcado como perdido, enquanto essa proporção é de aproximadamente 0,99% no grupo com operador identificado. Essa diferença indica uma forte associação entre a ausência de `operator_id` e o status de chamada perdida. Entretanto, os resultados, isoladamente, não permitem estabelecer uma relação causal nem concluir que os operadores identificados apresentam melhor desempenho. Uma hipótese a investigar é que chamadas não atendidas não sejam associadas a um operador específico pelo sistema.
- Alem do volume de chamadas, foi analisada a quantidade de registros em cada grupo. No grupo sem operador identificado, aproximadamente 99,1% dos registros estão marcados como perdidos. No grupo com operador identificado, essa proporção é de aproximadamente 5,4%. Essas porcentagens diferem das taxas calculadas anteriormente porque, nesta comparação, o denominador é a quantidade de registros, e não a soma de `calls_count`.
- Foi realizada uma análise descritiva de `calls_count` para os registros sem `operator_id`. A diferença entre a média(Nao perdidas - 9,50, Perdidas - 14,36) e a mediana(Nao perdidas - 2, Perdidas - 2) indica uma distribuição assimétrica de `calls_count`, especialmente entre as chamadas perdidas. Alguns registros com volumes elevados aumentam a média, enquanto a mediana permanece em 2. O valor máximo de 2.168 chamadas em um único registro merece investigação adicional. Entretanto, um valor elevado não constitui, por si só, evidência de erro, pois `calls_count` pode representar chamadas agregadas.
- Com base nos resultados obtidos até esta etapa, foram definidas as seguintes diretrizes:
    - Manter os registros sem `operator_id` nas métricas gerais de chamadas, pois eles representam parte relevante do volume observado.
    - Não preencher os valores ausentes de `operator_id` com identificadores arbitrarios.
    - Utilizar apenas registros com operador identificado nas analises de desempenho individual, deixando explícita essa limitação.
    - Evitar classificar operadores como ineficientes com base em taxas calculadas sobre volumes muito pequenos.
    - Investigar os registros com valores elevados de `calls_count` antes de decidir se existe algum problema de qualidade.
    - Não atribuir uma causa definitiva à diferença entre os grupos sem evidências adicionais sobre o funcionamento do sistema.
-A próxima etapa será investigar os registros com maior `calls_count` entre as chamadas recebidas perdidas sem operador identificado. O objetivo será verificar se existem padrões por usuário e data que ajudem a compreender esses valores elevados.

In [46]:
df_incoming[
    df_incoming['operator_id'].isna()
    & df_incoming['is_missed_call']
].nlargest(10, 'calls_count')[
    [
        'user_id',
        'date',
        'direction',
        'calls_count',
        'call_duration',
        'total_call_duration'
    ]
]

,user_id,date,direction,calls_count,call_duration,total_call_duration
37070,167626,2019-10-07 00:00:00+03:00,in,2168,0,2361
37102,167626,2019-10-08 00:00:00+03:00,in,1917,0,2044
37553,167626,2019-10-15 00:00:00+03:00,in,1914,0,2063
37308,167626,2019-10-11 00:00:00+03:00,in,1705,0,2455
50837,168252,2019-11-25 00:00:00+03:00,in,1679,0,42251
36913,167626,2019-10-04 00:00:00+03:00,in,1641,0,1210
50888,168252,2019-11-26 00:00:00+03:00,in,1590,0,42667
37193,167626,2019-10-09 00:00:00+03:00,in,1432,0,1273
37481,167626,2019-10-14 00:00:00+03:00,in,1411,0,1901
36824,167626,2019-10-03 00:00:00+03:00,in,1311,0,1136


In [49]:
df_incoming = df_incoming.assign(
    avg_total_duration_per_call=(
        df_incoming['total_call_duration']
        / df_incoming['calls_count']
    )
)

df_incoming[
    [
        'user_id',
        'date',
        'calls_count',
        'call_duration',
        'total_call_duration',
        'avg_total_duration_per_call'
    ]
]

,user_id,date,calls_count,call_duration,total_call_duration,avg_total_duration_per_call
0,166377,2019-08-04 00:00:00+03:00,2,0,4,2.000000
7,166377,2019-08-05 00:00:00+03:00,6,0,35,5.833333
9,166377,2019-08-06 00:00:00+03:00,4,0,62,15.500000
17,166377,2019-08-07 00:00:00+03:00,2,0,24,12.000000
26,166377,2019-08-12 00:00:00+03:00,1,407,411,411.000000
...,...,...,...,...,...,...
53889,168603,2019-11-28 00:00:00+03:00,1,0,30,30.000000
53891,168606,2019-11-08 00:00:00+03:00,1,0,7,7.000000
53893,168606,2019-11-08 00:00:00+03:00,6,0,121,20.166667
53894,168606,2019-11-08 00:00:00+03:00,2,686,705,352.500000


In [53]:
missed_without_operator = df_incoming[
    df_incoming['operator_id'].isna()
    & df_incoming['is_missed_call']
].copy()

# Calcular a duração média por chamada
missed_without_operator['avg_total_duration_per_call'] = (
    missed_without_operator['total_call_duration']
    / missed_without_operator['calls_count']
)

# Verificar as estatísticas descritivas
missed_without_operator['avg_total_duration_per_call'].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    7202.000000
mean       18.811097
std        21.750507
min         0.000000
50%        14.000000
75%        27.500000
90%        40.950758
95%        52.000000
99%        91.495000
max       465.000000
Name: avg_total_duration_per_call, dtype: float64

In [59]:
missed_without_operator = df_incoming[
    df_incoming['operator_id'].isna()
    & df_incoming['is_missed_call']
].copy()

missed_without_operator['avg_total_duration_per_call'] = (
    missed_without_operator['total_call_duration']
    / missed_without_operator['calls_count']
)

print('Total filtered records:',
      len(missed_without_operator))

print('Missing values ​​in calls_count:',
      missed_without_operator['calls_count'].isna().sum())

print('Missing values in total_call_duration:',
      missed_without_operator['total_call_duration'].isna().sum())

print('Values ​​valid for the average duration:',
      missed_without_operator['avg_total_duration_per_call'].count())

Total filtered records: 7202
Missing values ​​in calls_count: 0
Missing values in total_call_duration: 0
Values ​​valid for the average duration: 7202


In [56]:
missed_without_operator.nlargest(
    10, 'avg_total_duration_per_call'
)[
    [
        'user_id',
        'date',
        'calls_count',
        'call_duration',
        'total_call_duration',
        'avg_total_duration_per_call'
    ]
]

,user_id,date,calls_count,call_duration,total_call_duration,avg_total_duration_per_call
3044,166485,2019-09-22 00:00:00+03:00,1,0,465,465.0
19683,166983,2019-11-28 00:00:00+03:00,1,0,366,366.0
4220,166503,2019-10-27 00:00:00+03:00,1,0,354,354.0
5758,166536,2019-09-05 00:00:00+03:00,1,0,332,332.0
12629,166732,2019-10-30 00:00:00+03:00,2,0,588,294.0
19828,166997,2019-09-10 00:00:00+03:00,2,0,549,274.5
22291,167059,2019-11-01 00:00:00+03:00,1,0,225,225.0
28750,167185,2019-11-23 00:00:00+03:00,2,0,429,214.5
19647,166983,2019-11-26 00:00:00+03:00,2,0,426,213.0
28724,167185,2019-11-15 00:00:00+03:00,1,0,206,206.0


In [58]:
p99_duration = missed_without_operator[
    'avg_total_duration_per_call'
].quantile(0.99)

extreme_duration_records = missed_without_operator[
    missed_without_operator['avg_total_duration_per_call'] 
    > p99_duration
].copy()

print(f'99 percentile limit: {p99_duration:.2f}')
print(f'Records above the P99: {len(extreme_duration_records)}')
print(
    f'Records percentage: '
    f'{len(extreme_duration_records) / len(missed_without_operator) * 100:.2f}%'
)


99 percentile limit: 91.49
Records above the P99: 73
Records percentage: 1.01%


In [60]:
extreme_by_user = (
    extreme_duration_records
    .groupby('user_id')
    .agg(
        register_total=('user_id', 'size'),
        calls_total=('calls_count', 'sum'),
        max_avg_duration=(
            'avg_total_duration_per_call', 'max'
        )
    ).sort_values(
        ['register_total', 'max_avg_duration'],
        ascending=False
    )
)
extreme_by_user.head(10)

,register_total,calls_total,max_avg_duration
user_id,,,
166503,13,35,354.000000
166983,6,17,366.000000
167185,6,9,214.500000
166732,3,7,294.000000
167059,3,3,225.000000
166536,2,3,332.000000
166997,2,3,274.500000
167199,2,2,160.000000
167521,2,28,145.888889


In [61]:
top_recurrent_users = extreme_by_user.head(3).index

extreme_duration_records[
    extreme_duration_records['user_id'].isin(
        top_recurrent_users
    )
].sort_values(
    ['user_id', 'date']
)[
    [
        'user_id',
        'date',
        'calls_count',
        'call_duration',
        'total_call_duration',
        'avg_total_duration_per_call'
    ]
]

,user_id,date,calls_count,call_duration,total_call_duration,avg_total_duration_per_call
4050,166503,2019-09-21 00:00:00+03:00,2,0,218,109.000000
4161,166503,2019-10-13 00:00:00+03:00,4,0,549,137.250000
4180,166503,2019-10-17 00:00:00+03:00,4,0,762,190.500000
4191,166503,2019-10-20 00:00:00+03:00,2,0,314,157.000000
4216,166503,2019-10-25 00:00:00+03:00,4,0,447,111.750000
4218,166503,2019-10-26 00:00:00+03:00,1,0,115,115.000000
4220,166503,2019-10-27 00:00:00+03:00,1,0,354,354.000000
4243,166503,2019-11-03 00:00:00+03:00,2,0,210,105.000000
4252,166503,2019-11-05 00:00:00+03:00,2,0,364,182.000000
4268,166503,2019-11-10 00:00:00+03:00,3,0,338,112.666667


In [62]:
extreme_by_date = (
    extreme_duration_records
    .groupby('date')
    .agg(
        register_total=('user_id', 'size'),
        unique_users=('user_id', 'nunique'),
        calls_total=('calls_count', 'sum'),
        max_avg_duration=(
            'avg_total_duration_per_call', 'max'
        )
    ).sort_values(
        ['register_total', 'max_avg_duration'],
        ascending=False
    )
)

extreme_by_date.head(10)

,register_total,unique_users,calls_total,max_avg_duration
date,,,,
2019-10-25 00:00:00+03:00,5,5,8,141.000000
2019-11-17 00:00:00+03:00,4,4,7,108.000000
2019-10-30 00:00:00+03:00,3,3,4,294.000000
2019-11-26 00:00:00+03:00,3,3,10,213.000000
2019-11-15 00:00:00+03:00,3,3,4,206.000000
2019-11-12 00:00:00+03:00,3,3,7,124.333333
2019-11-24 00:00:00+03:00,3,3,12,120.500000
2019-11-01 00:00:00+03:00,2,2,2,225.000000
2019-11-23 00:00:00+03:00,2,2,3,214.500000


In [64]:
extreme_records_by_date = (
    extreme_duration_records
    .groupby('date')
    .size()
    .sort_values(ascending=False)
)

print('Total number of extreme records:',
      len(extreme_duration_records))

print('Different dates:',
      extreme_duration_records['date'].nunique())

print('Maximum number of extreme records on a given date:',
      extreme_records_by_date.max())

print('\nDistribution of the number of records by date:')
print(extreme_records_by_date.value_counts().sort_index())

Total number of extreme records: 73
Different dates: 47
Maximum number of extreme records on a given date: 5

Distribution of the number of records by date:
1    31
2     9
3     5
4     1
5     1
Name: count, dtype: int64


### Investigacao de valores extremos na duracao media por chamada
- O objetivo era investigar os registros com duracao total media por chamada elevada entre as chamadas recebidas perdidas sem operador identificado `operator_id ausente`, verificando a distribuicao dos valores e a possivel concentracao por usuario e data. 